### **Gemma4-E2B-it Model Inspection**

In [43]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, AutoConfig
import torch
from typing import Any, List
from pathlib import Path
import json

In [42]:
ROOT = Path()
output_path = Path("artifacts")
output_path.mkdir(exist_ok=True)

In [12]:
processor = AutoProcessor.from_pretrained("google/gemma-4-E2B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-E2B-it", dtype=torch.bfloat16, attn_implementation="eager")

Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 2889.32it/s]


In [46]:
def write_json(data: Any, destination: Path):
    path = output_path / destination

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)
        bytes_written = file.tell()

    print(f"Number of bytes written: {bytes_written}")

### **Text Config**

In [47]:
write_json(model.config.get_text_config().to_dict(), Path("text_config.json"))

Number of bytes written: 2824


### **Tensor Map**

In [62]:
tensor_map = []
vision_tensors = 0
audio_tensors = 0
text_tensors = 0

for name, tensor in model.state_dict().items():

    if "vision" in name:
        vision_tensors += 1
    elif "audio" in name:
        audio_tensors += 1
    else:
        text_tensors += 1

        tensor_map.append({
            "name": name,
            "shape": tuple(tensor.shape),
            "dtype": str(tensor.dtype)
        })

write_json(tensor_map, Path("tensor_map.json"))

print(f"\nVision Tensors: {vision_tensors}")
print(f"Audio Tensors: {audio_tensors}")
print(f"Text tensors: {text_tensors}")

Number of bytes written: 96206

Vision Tensors: 659
Audio Tensors: 752
Text tensors: 541


In [6]:
prompt = "I am Ojas"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt}
        ]
    }
]

In [7]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

In [8]:
output = model.generate(**inputs, max_new_tokens=50, cache_implementation="static")
print(processor.decode(output[0][input_len:], skip_special_tokens=True))

Hi Ojas, it's nice to meet you! How can I help you today? 😊
